# 06 — End-to-End Preparation for Machine Learning

## Task 1 — Alzheimer binary classification

This notebook integrates the preparation decisions introduced in notebooks 01–05 into one reproducible workflow that starts from the raw tabular file and ends with model-ready TRAIN, VALIDATION, and TEST matrices.

The notebook **does not train a predictive model**. Its purpose is to freeze the preparation boundary that later model-training notebooks can reuse.

The central rule is:

```text
split first
fit data-dependent preparation on TRAIN only
transform VALIDATION and TEST without refitting
```

The workflow also exports a fitted preparation object and demonstrates that it can be reloaded and applied to a target-free held-out batch.


## Learning objectives

By the end of this notebook, you should be able to:

- connect deterministic preprocessing, feature engineering, splitting, scaling, and feature selection;
- distinguish operations that are fixed by a data contract from operations learned from data;
- construct a reusable scikit-learn preparation object;
- fit all data-dependent transformations using TRAIN only;
- transform VALIDATION and TEST with the frozen preparation;
- export model-ready split tables and reproducibility metadata;
- serialize both an unfitted preparation template and the fitted preparation artifact;
- reload the fitted artifact and prepare new target-free observations;
- understand why cross-validation later requires refitting the preparation workflow inside each training fold.


## 1. Workflow decision for this task

This binary-classification demonstration uses a **stratified** split because the class proportion is part of the supervised problem. The target is already encoded as 0/1, so no separate target encoder is required. Numerical predictors are standardized; binary 0/1 indicators keep their interpretation.

The 70/15/15 split used here is a **teaching choice for the upcoming fixed-split training/validation/testing demonstrations**. It is not a universal recommendation. When data are scarce, a held-out TEST set plus cross-validation inside TRAIN may use the available observations more efficiently.


## 2. Imports and reproducibility

This is the first tabular-data notebook in the sequence where using a scikit-learn `Pipeline` is intentional. Earlier notebooks separated the techniques so that each operation could be inspected individually. Here the goal is different: preserve the fitted preparation as one reusable artifact.

The pipeline created below contains **preparation only**. No estimator is attached.


In [1]:
from pathlib import Path
import json
import re
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
SELECTION_FRACTION = 0.60

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)


Python: 3.14.8
pandas: 3.0.6
scikit-learn: 1.9.1
joblib: 1.6.0


## 3. Locate the project and define outputs

Generated model-ready tables are written under the dataset directory. Reusable preparation artifacts and manifests are written under `results/`.

These generated outputs are products of executing the notebook; the notebook itself remains the reproducible source of the workflow.


In [2]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / "data" / "synthetic_alzheimer"
RAW_DIR = DATASET_DIR / "raw_data"
RAW_DATA_PATH = RAW_DIR / "synthetic_alzheimer_patients_1000.csv"
METADATA_PATH = RAW_DIR / "metadata.json"

ML_READY_DIR = DATASET_DIR / "ml_ready"
RESULTS_DIR = PROJECT_ROOT / "results" / "synthetic_alzheimer" / "ml_preparation"
ARTIFACTS_DIR = RESULTS_DIR / "artifacts"
TABLES_DIR = RESULTS_DIR / "tables"

for directory in [ML_READY_DIR, ARTIFACTS_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_PATH.relative_to(PROJECT_ROOT))
print("ML-ready output:", ML_READY_DIR.relative_to(PROJECT_ROOT))
print("Preparation artifacts:", ARTIFACTS_DIR.relative_to(PROJECT_ROOT))


Project root: /home/david/Desktop/depto_computacion/ml_2026/demos_machine_learning_2026
Raw data: data/synthetic_alzheimer/raw_data/synthetic_alzheimer_patients_1000.csv
ML-ready output: data/synthetic_alzheimer/ml_ready
Preparation artifacts: results/synthetic_alzheimer/ml_preparation/artifacts


## 4. Load the raw table and metadata

The end-to-end workflow starts from the raw table instead of assuming that an earlier notebook has already been executed. The metadata acts as the input contract for column roles.


In [3]:
with METADATA_PATH.open("r", encoding="utf-8") as file:
    metadata = json.load(file)

raw_df = pd.read_csv(RAW_DATA_PATH)

print("Raw shape:", raw_df.shape)
display(raw_df.head())
print("Metadata target:", metadata["label"])
print("Metadata identifier:", metadata["col_id"])


Raw shape: (1000, 31)


,patient_id,age,sex_female,height_cm,weight_kg,bmi,waist_cm,hip_cm,waist_hip_ratio,systolic_bp_mmHg,...,traumatic_brain_injury,family_history_alzheimer,sleep_disorder,chronic_kidney_disease,hearing_loss,asthma,allergy_medication,allergy_food,allergy_environmental,alzheimer
0,P0001,75,1,158.7,69.5,27.6,116.6,136.3,0.855,137,...,1,0,0,0,0,0,0,0,0,1
1,P0002,50,1,156.5,70.3,28.7,122.1,130.5,0.936,99,...,0,0,0,0,0,0,0,0,0,1
2,P0003,82,0,169.4,76.6,26.7,117.8,118.3,0.996,153,...,0,1,1,0,0,0,0,1,0,1
3,P0004,72,1,165.2,80.8,29.6,110.1,123.4,0.892,121,...,0,1,0,0,0,0,0,0,1,1
4,P0005,64,1,151.0,75.4,33.0,127.2,144.4,0.881,153,...,0,1,1,0,0,0,0,0,1,0


Metadata target: alzheimer
Metadata identifier: patient_id


## 5. Deterministic structural preprocessing

The following function performs only contract-based operations:

- normalize column names;
- validate required columns;
- remove exact duplicate rows;
- require unique, non-missing identifiers;
- coerce declared predictors to numerical dtype;
- validate non-missing binary indicators as 0/1;
- validate the target when it is present.

Predictor missing values are **not imputed here**. Imputation is data-dependent and therefore belongs in the fitted training-only preparation object.


In [4]:
def normalize_column_name(name: str) -> str:
    name = str(name).strip().lower()
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name)
    return re.sub(r"_+", "_", name).strip("_")


def normalized_roles(metadata: dict) -> dict:
    return {
        "id": normalize_column_name(metadata["col_id"]),
        "target": normalize_column_name(metadata["label"]),
        "numerical": [normalize_column_name(c) for c in metadata["numerical_column"]],
        "categorical": [normalize_column_name(c) for c in metadata["categorical_column"]],
    }


ROLES = normalized_roles(metadata)
ID_COLUMN = ROLES["id"]
TARGET_COLUMN = ROLES["target"]
ORIGINAL_NUMERICAL = ROLES["numerical"]
CATEGORICAL_COLUMNS = ROLES["categorical"]


def clean_raw_table(frame: pd.DataFrame, require_target: bool = True) -> pd.DataFrame:
    out = frame.copy()
    out.columns = [normalize_column_name(c) for c in out.columns]

    required = [ID_COLUMN, *ORIGINAL_NUMERICAL, *CATEGORICAL_COLUMNS]
    if require_target:
        required.append(TARGET_COLUMN)

    missing_columns = sorted(set(required) - set(out.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    out = out.drop_duplicates().copy()

    if out[ID_COLUMN].isna().any():
        raise ValueError("Identifier contains missing values.")
    if out[ID_COLUMN].duplicated().any():
        raise ValueError("Identifier must be unique after exact-duplicate removal.")

    for column in [*ORIGINAL_NUMERICAL, *CATEGORICAL_COLUMNS]:
        out[column] = pd.to_numeric(out[column], errors="raise")

    for column in CATEGORICAL_COLUMNS:
        observed = set(out[column].dropna().unique().tolist())
        if not observed.issubset({0, 1}):
            raise ValueError(f"{column} contains values outside {0, 1}: {observed}")

    if TARGET_COLUMN in out.columns:
        if True:
            out[TARGET_COLUMN] = pd.to_numeric(out[TARGET_COLUMN], errors="raise")
        else:
            out[TARGET_COLUMN] = out[TARGET_COLUMN].astype("string").str.strip()

        if require_target and out[TARGET_COLUMN].isna().any():
            raise ValueError("Training/evaluation data cannot contain missing target values.")

    return out


clean_df = clean_raw_table(raw_df, require_target=True)
print("Clean shape:", clean_df.shape)
print("Exact duplicates after cleaning:", int(clean_df.duplicated().sum()))
print("Duplicated identifiers:", int(clean_df[ID_COLUMN].duplicated().sum()))
print("Predictor missing values retained for fitted imputation:",
      int(clean_df[ORIGINAL_NUMERICAL + CATEGORICAL_COLUMNS].isna().sum().sum()))


Clean shape: (1000, 31)
Exact duplicates after cleaning: 0
Duplicated identifiers: 0
Predictor missing values retained for fitted imputation: 0


## 6. Deterministic feature engineering

The feature recipe is fixed before the split and does not estimate statistics from the observed sample. The same function will therefore be applied to TRAIN, VALIDATION, TEST, and future external batches.

If this workflow were promoted from teaching material to a production system, this deterministic function would ideally live in tested source code rather than only inside a notebook.


In [5]:
def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    height = out["height_cm"].replace(0, np.nan)

    out["pulse_pressure_mmhg"] = out["systolic_bp_mmhg"] - out["diastolic_bp_mmhg"]
    out["mean_arterial_pressure_mmhg"] = (
        out["systolic_bp_mmhg"] + 2.0 * out["diastolic_bp_mmhg"]
    ) / 3.0
    out["waist_height_ratio"] = out["waist_cm"] / height

    cardiometabolic = [
        "hypertension", "diabetes_type2", "dyslipidemia",
        "cardiovascular_disease", "stroke_history",
    ]
    neurobehavioral = [
        "depression_history", "traumatic_brain_injury",
        "sleep_disorder", "hearing_loss",
    ]

    out["cardiometabolic_risk_count"] = out[cardiometabolic].sum(axis=1, min_count=1)
    out["neurobehavioral_risk_count"] = out[neurobehavioral].sum(axis=1, min_count=1)
    out = out.replace([np.inf, -np.inf], np.nan)
    return out


In [6]:
ENGINEERED_NUMERICAL = ['pulse_pressure_mmhg', 'mean_arterial_pressure_mmhg', 'waist_height_ratio', 'cardiometabolic_risk_count', 'neurobehavioral_risk_count']
NUMERICAL_FEATURES = ORIGINAL_NUMERICAL + ENGINEERED_NUMERICAL
FEATURE_COLUMNS = NUMERICAL_FEATURES + CATEGORICAL_COLUMNS

preview_eng = engineer_features(clean_df.head(5))
assert set(FEATURE_COLUMNS).issubset(preview_eng.columns)

feature_roles = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "role": ["numerical"] * len(NUMERICAL_FEATURES) + ["binary_categorical"] * len(CATEGORICAL_COLUMNS),
    "origin": (
        ["original"] * len(ORIGINAL_NUMERICAL)
        + ["engineered"] * len(ENGINEERED_NUMERICAL)
        + ["original"] * len(CATEGORICAL_COLUMNS)
    ),
})
display(feature_roles)
print("Candidate predictors:", len(FEATURE_COLUMNS))


,feature,role,origin
0,age,numerical,original
1,height_cm,numerical,original
2,weight_kg,numerical,original
3,bmi,numerical,original
4,waist_cm,numerical,original
5,hip_cm,numerical,original
6,waist_hip_ratio,numerical,original
7,systolic_bp_mmhg,numerical,original
8,diastolic_bp_mmhg,numerical,original
9,heart_rate_bpm,numerical,original


Candidate predictors: 34


## 7. Create TRAIN, VALIDATION, and TEST before fitting preparation

All fitted preprocessing decisions must occur **after** this cell. The identifier is used only to audit the partition and is never passed as a predictor.


In [7]:
train_df, temporary_df = train_test_split(
    clean_df,
    test_size=(VALIDATION_FRACTION + TEST_FRACTION),
    random_state=RANDOM_STATE,
    stratify=clean_df[TARGET_COLUMN],
)
validation_df, test_df = train_test_split(
    temporary_df,
    test_size=TEST_FRACTION / (VALIDATION_FRACTION + TEST_FRACTION),
    random_state=RANDOM_STATE,
    stratify=temporary_df[TARGET_COLUMN],
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)


In [8]:
assert set(train_df[ID_COLUMN]).isdisjoint(validation_df[ID_COLUMN])
assert set(train_df[ID_COLUMN]).isdisjoint(test_df[ID_COLUMN])
assert set(validation_df[ID_COLUMN]).isdisjoint(test_df[ID_COLUMN])
assert len(train_df) + len(validation_df) + len(test_df) == len(clean_df)

split_summary = pd.DataFrame({
    "split": ["TRAIN", "VALIDATION", "TEST"],
    "n": [len(train_df), len(validation_df), len(test_df)],
    "fraction": [len(train_df)/len(clean_df), len(validation_df)/len(clean_df), len(test_df)/len(clean_df)],
})
display(split_summary)


,split,n,fraction
0,TRAIN,700,0.70
1,VALIDATION,150,0.15
2,TEST,150,0.15


## 8. Apply the deterministic feature recipe to each partition

This step uses the same formula definitions everywhere. It does not learn from VALIDATION or TEST.


In [9]:
train_eng = engineer_features(train_df)
validation_eng = engineer_features(validation_df)
test_eng = engineer_features(test_df)

X_train = train_eng[FEATURE_COLUMNS].copy()
X_validation = validation_eng[FEATURE_COLUMNS].copy()
X_test = test_eng[FEATURE_COLUMNS].copy()

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)


X_train: (700, 34)
X_validation: (150, 34)
X_test: (150, 34)


## 9. Prepare the target

Predictor preparation and target representation are separate concerns. The target transformation used below is task-specific and is saved when future interpretation requires an inverse mapping.


In [10]:
y_train = train_df[TARGET_COLUMN].astype(int).to_numpy()
y_validation = validation_df[TARGET_COLUMN].astype(int).to_numpy()
y_test = test_df[TARGET_COLUMN].astype(int).to_numpy()
y_train_for_preparation = y_train

print("TRAIN class prevalence:")
display(train_df[TARGET_COLUMN].value_counts(normalize=True).sort_index().rename("proportion").to_frame())


TRAIN class prevalence:


,proportion
alzheimer,
0,0.657143
1,0.342857


## 10. Define an unfitted reusable preparation template

The template contains only scikit-learn objects that can be serialized safely without depending on notebook-local transformation functions.

The deterministic feature recipe remains explicit in this notebook. The fitted pipeline begins at the candidate-feature table and performs:

```text
numerical predictors
    median imputation
    StandardScaler

binary predictors
    most-frequent imputation

all prepared predictors
    supervised SelectKBest
```

The selector retains 60% of the candidate predictors for this teaching example. That fraction is not presented as an optimum; later modelling can compare different preparation choices.


In [11]:
SELECTION_K = max(1, int(np.ceil(SELECTION_FRACTION * len(FEATURE_COLUMNS))))

numerical_preparation = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

binary_preparation = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

column_preparation = ColumnTransformer(
    transformers=[
        ("numerical", numerical_preparation, NUMERICAL_FEATURES),
        ("binary", binary_preparation, CATEGORICAL_COLUMNS),
    ],
    remainder="drop",
    sparse_threshold=0.0,
    verbose_feature_names_out=False,
)

preparation_template = Pipeline([
    ("columns", column_preparation),
    ("selector", SelectKBest(score_func=f_classif, k=SELECTION_K)),
])

print(preparation_template)
print(f"Selector retains {SELECTION_K} of {len(FEATURE_COLUMNS)} candidate predictors.")


Pipeline(steps=[('columns',
                 ColumnTransformer(sparse_threshold=0.0,
                                   transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'height_cm',
                                                   'weight_kg', 'bmi',
                                                   'waist_cm', 'hip_cm',
                                                   'waist_hip_ratio',
                                                   'systolic_bp_mmhg',
                                                   'diastolic_bp_mmhg',
                                                   'heart_rate_bpm',
          

## 11. Save the unfitted template

The unfitted template is useful later if a modelling notebook performs cross-validation. It can be cloned or combined with an estimator so the complete preparation is refitted inside every training fold.


In [12]:
TEMPLATE_PATH = ARTIFACTS_DIR / "preparation_pipeline_template.joblib"
joblib.dump(clone(preparation_template), TEMPLATE_PATH)
print("Saved:", TEMPLATE_PATH.relative_to(PROJECT_ROOT))


Saved: results/synthetic_alzheimer/ml_preparation/artifacts/preparation_pipeline_template.joblib


## 12. Fit preparation using TRAIN only

This is the fitting boundary. Imputation statistics, scaling parameters, and supervised feature-selection scores are learned here and nowhere else.


In [13]:
preparation_fitted = clone(preparation_template)
preparation_fitted.fit(X_train, y_train_for_preparation)

X_train_ready = preparation_fitted.transform(X_train)
X_validation_ready = preparation_fitted.transform(X_validation)
X_test_ready = preparation_fitted.transform(X_test)

pre_feature_names = preparation_fitted.named_steps["columns"].get_feature_names_out()
selection_mask = preparation_fitted.named_steps["selector"].get_support()
SELECTED_FEATURES = pre_feature_names[selection_mask].tolist()

assert X_train_ready.shape[1] == len(SELECTED_FEATURES)
assert X_validation_ready.shape[1] == len(SELECTED_FEATURES)
assert X_test_ready.shape[1] == len(SELECTED_FEATURES)
assert np.isfinite(X_train_ready).all()
assert np.isfinite(X_validation_ready).all()
assert np.isfinite(X_test_ready).all()

print("Selected predictors:", len(SELECTED_FEATURES))
display(pd.DataFrame({"selected_feature": SELECTED_FEATURES}))


Selected predictors: 21


,selected_feature
0,age
1,waist_hip_ratio
2,systolic_bp_mmhg
3,diastolic_bp_mmhg
4,pulse_pressure_mmhg
5,mean_arterial_pressure_mmhg
6,cardiometabolic_risk_count
7,neurobehavioral_risk_count
8,physical_inactivity
9,hypertension


## 13. Inspect training-derived fitted parameters

This cell is an audit, not another fitting step. The values shown below were learned exclusively from TRAIN.


In [14]:
num_pipeline_fitted = preparation_fitted.named_steps["columns"].named_transformers_["numerical"]
imputer_statistics = pd.Series(
    num_pipeline_fitted.named_steps["imputer"].statistics_,
    index=NUMERICAL_FEATURES,
    name="training_median",
)
scaler_means = pd.Series(
    num_pipeline_fitted.named_steps["scaler"].mean_,
    index=NUMERICAL_FEATURES,
    name="training_mean_after_imputation",
)
scaler_scales = pd.Series(
    num_pipeline_fitted.named_steps["scaler"].scale_,
    index=NUMERICAL_FEATURES,
    name="training_scale",
)

fitted_numeric_parameters = pd.concat(
    [imputer_statistics, scaler_means, scaler_scales], axis=1
)
display(fitted_numeric_parameters.head(12))


,training_median,training_mean_after_imputation,training_scale
age,69.000000,68.640000,10.925807
height_cm,165.300000,165.481286,8.319442
weight_kg,73.000000,74.176286,13.586442
bmi,27.100000,27.032286,4.173650
waist_cm,118.100000,118.069429,8.976223
hip_cm,127.750000,127.706000,10.035007
waist_hip_ratio,0.924000,0.925553,0.038865
systolic_bp_mmhg,130.000000,130.350000,14.177006
diastolic_bp_mmhg,80.000000,79.885714,8.317525
heart_rate_bpm,72.000000,71.647143,8.167868


## 14. Export model-ready fixed splits

The transformed tables are convenient for the upcoming **fixed train/validation/test modelling demonstrations**. They include the identifier for traceability but the identifier is not part of the prepared predictor matrix.

For cross-validation, prefer loading the unfitted preparation template and refitting it inside each fold instead of cross-validating directly on a matrix produced by a selector already fitted to the full TRAIN partition.


In [15]:
def attach_target(frame, split: str):
    out = frame.copy()
    target_values = {
        "train": y_train,
        "validation": y_validation,
        "test": y_test,
    }[split]
    out[TARGET_COLUMN] = target_values
    return out


In [16]:
def ready_frame(ids, matrix, selected_features):
    out = pd.DataFrame(matrix, columns=selected_features)
    out.insert(0, ID_COLUMN, ids.reset_index(drop=True))
    return out

train_ready_df = ready_frame(train_df[ID_COLUMN], X_train_ready, SELECTED_FEATURES)
validation_ready_df = ready_frame(validation_df[ID_COLUMN], X_validation_ready, SELECTED_FEATURES)
test_ready_df = ready_frame(test_df[ID_COLUMN], X_test_ready, SELECTED_FEATURES)

train_ready_df = attach_target(train_ready_df, split="train")
validation_ready_df = attach_target(validation_ready_df, split="validation")
test_ready_df = attach_target(test_ready_df, split="test")

TRAIN_READY_PATH = ML_READY_DIR / "train_ready.csv"
VALIDATION_READY_PATH = ML_READY_DIR / "validation_ready.csv"
TEST_READY_PATH = ML_READY_DIR / "test_ready.csv"

train_ready_df.to_csv(TRAIN_READY_PATH, index=False)
validation_ready_df.to_csv(VALIDATION_READY_PATH, index=False)
test_ready_df.to_csv(TEST_READY_PATH, index=False)

print("Exported model-ready tables:")
for path in [TRAIN_READY_PATH, VALIDATION_READY_PATH, TEST_READY_PATH]:
    print(" -", path.relative_to(PROJECT_ROOT))

display(train_ready_df.head())


Exported model-ready tables:
 - data/synthetic_alzheimer/ml_ready/train_ready.csv
 - data/synthetic_alzheimer/ml_ready/validation_ready.csv
 - data/synthetic_alzheimer/ml_ready/test_ready.csv


,patient_id,age,waist_hip_ratio,systolic_bp_mmhg,diastolic_bp_mmhg,pulse_pressure_mmhg,mean_arterial_pressure_mmhg,cardiometabolic_risk_count,neurobehavioral_risk_count,physical_inactivity,...,cardiovascular_disease,stroke_history,depression_history,traumatic_brain_injury,family_history_alzheimer,chronic_kidney_disease,hearing_loss,allergy_medication,allergy_food,alzheimer
0,P0132,0.673634,0.526107,0.962827,-0.226716,1.051632,0.409631,0.908842,-0.872406,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,P0232,0.856687,0.165886,-0.377372,-1.428996,0.442411,-1.207569,0.908842,0.432475,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1
2,P0460,-0.790788,-0.400177,-1.153276,-0.827856,-0.640650,-1.249035,-0.998772,-0.872406,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,P0624,0.124476,-0.683208,0.045849,-0.827856,0.510102,-0.544102,1.862648,0.432475,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0
4,P0325,-1.339947,-0.322986,-1.012202,-0.346944,-0.776032,-0.834369,-0.998772,-0.872406,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


## 15. Export split assignments

A random seed helps reproduce a split, but an explicit assignment table is a stronger record of which observations were used for each role.


In [17]:
split_assignments = pd.concat([
    train_df[[ID_COLUMN, TARGET_COLUMN]].assign(split="TRAIN"),
    validation_df[[ID_COLUMN, TARGET_COLUMN]].assign(split="VALIDATION"),
    test_df[[ID_COLUMN, TARGET_COLUMN]].assign(split="TEST"),
], ignore_index=True)

SPLIT_ASSIGNMENTS_PATH = TABLES_DIR / "split_assignments.csv"
split_assignments.to_csv(SPLIT_ASSIGNMENTS_PATH, index=False)
display(split_assignments.head())


,patient_id,alzheimer,split
0,P0132,0,TRAIN
1,P0232,1,TRAIN
2,P0460,0,TRAIN
3,P0624,0,TRAIN
4,P0325,0,TRAIN


## 16. Save the fitted preparation artifact

This fitted object is the frozen predictor preparation learned from TRAIN. It can transform VALIDATION, TEST, and later external observations, provided the deterministic feature recipe and input schema are respected.


In [18]:
FITTED_PIPELINE_PATH = ARTIFACTS_DIR / "preparation_pipeline_fitted.joblib"
joblib.dump(preparation_fitted, FITTED_PIPELINE_PATH)
print("Saved:", FITTED_PIPELINE_PATH.relative_to(PROJECT_ROOT))


Saved: results/synthetic_alzheimer/ml_preparation/artifacts/preparation_pipeline_fitted.joblib


## 17. Export the preparation manifest

The serialized object is useful for computation; the manifest is useful for auditability. It records the choices needed to understand what the object expects and what it produces.


In [19]:
manifest = {
    "dataset": "synthetic_alzheimer",
    "task_type": "binary_classification",
    "source": str(RAW_DATA_PATH.relative_to(PROJECT_ROOT)),
    "identifier": ID_COLUMN,
    "target": TARGET_COLUMN,
    "random_state": RANDOM_STATE,
    "split_fractions": {"train": TRAIN_FRACTION, "validation": VALIDATION_FRACTION, "test": TEST_FRACTION},
    "split_sizes": {"train": len(train_df), "validation": len(validation_df), "test": len(test_df)},
    "original_numerical_features": ORIGINAL_NUMERICAL,
    "engineered_numerical_features": ENGINEERED_NUMERICAL,
    "binary_categorical_features": CATEGORICAL_COLUMNS,
    "candidate_features": FEATURE_COLUMNS,
    "selection_method": "SelectKBest(f_classif)",
    "selection_fraction": SELECTION_FRACTION,
    "selection_k": SELECTION_K,
    "selected_features": SELECTED_FEATURES,
    "prepared_dimensions": {
        "train": list(X_train_ready.shape),
        "validation": list(X_validation_ready.shape),
        "test": list(X_test_ready.shape),
    },
    "artifacts": {
        "unfitted_preparation_template": str(TEMPLATE_PATH.relative_to(PROJECT_ROOT)),
        "fitted_preparation": str(FITTED_PIPELINE_PATH.relative_to(PROJECT_ROOT)),
        "split_assignments": str(SPLIT_ASSIGNMENTS_PATH.relative_to(PROJECT_ROOT)),
    },
    "software": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
    "fixed_split_note": (
        "The fitted preparation artifact is valid for the fixed TRAIN/VALIDATION/TEST workflow "
        "and for later target-free external observations transformed without refitting."
    ),
    "cross_validation_warning": (
        "If later model development uses cross-validation, load/rebuild the UNFITTED preparation "
        "template and fit it inside each training fold together with the estimator. Do not perform "
        "cross-validation on features selected once using the complete TRAIN partition."
    ),
}



MANIFEST_PATH = RESULTS_DIR / "preparation_manifest.json"
with MANIFEST_PATH.open("w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)

SELECTED_FEATURES_PATH = RESULTS_DIR / "selected_feature_names.json"
with SELECTED_FEATURES_PATH.open("w", encoding="utf-8") as file:
    json.dump(SELECTED_FEATURES, file, indent=2)

print("Saved:", MANIFEST_PATH.relative_to(PROJECT_ROOT))
print("Saved:", SELECTED_FEATURES_PATH.relative_to(PROJECT_ROOT))


Saved: results/synthetic_alzheimer/ml_preparation/preparation_manifest.json
Saved: results/synthetic_alzheimer/ml_preparation/selected_feature_names.json


## 18. Reload the fitted artifact and prepare target-free external data

A reusable preparation workflow should work when the target is unavailable. To test that boundary, the next cells reconstruct a small batch from held-out raw observations, remove the target, run the same deterministic cleaning and feature recipe, reload the preparation object from disk, and call **`transform()` only**.

No parameter is refitted on this batch.


In [20]:
external_ids = test_df[ID_COLUMN].head(5).tolist()
raw_id_column = metadata["col_id"]
raw_target_column = metadata["label"]

external_raw = (
    raw_df.set_index(raw_id_column)
    .loc[external_ids]
    .reset_index()
    .drop(columns=[raw_target_column])
)

external_clean = clean_raw_table(external_raw, require_target=False)
external_eng = engineer_features(external_clean)
external_X = external_eng[FEATURE_COLUMNS]

loaded_preparation = joblib.load(FITTED_PIPELINE_PATH)
external_ready = loaded_preparation.transform(external_X)

np.testing.assert_allclose(
    external_ready,
    X_test_ready[: len(external_ids)],
    rtol=1e-10,
    atol=1e-10,
)

external_ready_df = pd.DataFrame(external_ready, columns=SELECTED_FEATURES)
external_ready_df.insert(0, ID_COLUMN, external_ids)

display(external_ready_df)
print("Reload + external transform check: PASS")


,patient_id,age,waist_hip_ratio,systolic_bp_mmhg,diastolic_bp_mmhg,pulse_pressure_mmhg,mean_arterial_pressure_mmhg,cardiometabolic_risk_count,neurobehavioral_risk_count,physical_inactivity,...,dyslipidemia,cardiovascular_disease,stroke_history,depression_history,traumatic_brain_injury,family_history_alzheimer,chronic_kidney_disease,hearing_loss,allergy_medication,allergy_food
0,P0589,1.131267,-0.786128,1.174437,-1.308768,1.863928,-0.212369,-0.044965,0.432475,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,P0894,-0.150103,1.092169,0.045849,-0.467172,0.307028,-0.295302,-0.044965,0.432475,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,P0408,0.399055,1.581041,1.597657,-0.467172,1.796236,0.616965,1.862648,0.432475,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,P0839,-0.058577,0.140156,1.597657,1.576705,0.645485,2.026832,0.908842,1.737355,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
4,P0834,0.765161,0.577567,-0.377372,0.735109,-0.776032,0.285231,2.816455,0.432475,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


Reload + external transform check: PASS


## 19. Final validation

Before handing these outputs to a modelling notebook, verify the preparation contract:

- partitions are disjoint;
- preparation was fitted once using TRAIN only;
- VALIDATION and TEST were transformed without refitting;
- the prepared matrices contain no missing or infinite values;
- identifiers are not predictors;
- selected feature names are frozen;
- the fitted object survives serialization and reload;
- a target-free held-out batch produces the same prepared representation;
- cross-validation later will use the unfitted template rather than the already selected TRAIN matrix.


In [21]:
checks = {
    "train_validation_disjoint": set(train_df[ID_COLUMN]).isdisjoint(validation_df[ID_COLUMN]),
    "train_test_disjoint": set(train_df[ID_COLUMN]).isdisjoint(test_df[ID_COLUMN]),
    "validation_test_disjoint": set(validation_df[ID_COLUMN]).isdisjoint(test_df[ID_COLUMN]),
    "train_ready_finite": bool(np.isfinite(X_train_ready).all()),
    "validation_ready_finite": bool(np.isfinite(X_validation_ready).all()),
    "test_ready_finite": bool(np.isfinite(X_test_ready).all()),
    "identifier_excluded": ID_COLUMN not in SELECTED_FEATURES,
    "target_excluded": TARGET_COLUMN not in SELECTED_FEATURES,
    "serialized_pipeline_exists": FITTED_PIPELINE_PATH.exists(),
    "template_exists": TEMPLATE_PATH.exists(),
    "manifest_exists": MANIFEST_PATH.exists(),
}

validation_table = pd.DataFrame({"check": checks.keys(), "passed": checks.values()})
display(validation_table)

if not all(checks.values()):
    raise AssertionError("At least one final preparation check failed.")

print("FINAL PREPARATION VALIDATION: PASS")


,check,passed
0,train_validation_disjoint,True
1,train_test_disjoint,True
2,validation_test_disjoint,True
3,train_ready_finite,True
4,validation_ready_finite,True
5,test_ready_finite,True
6,identifier_excluded,True
7,target_excluded,True
8,serialized_pipeline_exists,True
9,template_exists,True


FINAL PREPARATION VALIDATION: PASS


# Final interpretation

This notebook has frozen a **data-preparation boundary**, not a predictive model.

The next modelling notebooks can now take two complementary routes:

### Fixed split teaching workflow

Use the exported `train_ready.csv`, `validation_ready.csv`, and `test_ready.csv` files to demonstrate model fitting, model selection, and final testing with a simple fixed partition.

### Cross-validation workflow

Load the **unfitted** preparation template and combine it with an estimator. The preparation and estimator should then be fitted together inside every training fold.

For future external prediction, the eventual trained model must receive observations that pass through the same deterministic feature recipe and the same fitted preparation logic used during model development.
